# Week 2 - Task 3: Data Cleaning
Real-world data is messy. In this notebook the raw student dataset (`students_raw.csv`) is checked for **missing values, duplicate records and wrong data types**, cleaned, and saved as `students_clean.csv`.

In [1]:
import pandas as pd
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 20)
raw = pd.read_csv("students_raw.csv")
df = raw.copy()   # keep the original untouched for comparison

## A. Before cleaning - dataset analysis

In [2]:
print("Shape:", df.shape)
df.info()

Shape: (126, 9)
<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Student_ID   126 non-null    int64  
 1   Name         126 non-null    str    
 2   Gender       126 non-null    str    
 3   Age          122 non-null    float64
 4   Study_Hours  122 non-null    float64
 5   Attendance   121 non-null    str    
 6   Math         120 non-null    float64
 7   Science      121 non-null    float64
 8   English      121 non-null    float64
dtypes: float64(5), int64(1), str(3)
memory usage: 9.0 KB


In [3]:
print("Missing values per column:")
df.isnull().sum()

Missing values per column:


Student_ID     0
Name           0
Gender         0
Age            4
Study_Hours    4
Attendance     5
Math           6
Science        5
English        5
dtype: int64

In [4]:
print("Total missing values   :", df.isnull().sum().sum())
print("Duplicate rows         :", df.duplicated().sum())
print("\nThe duplicate records:")
df[df.duplicated(keep=False)].sort_values("Student_ID")

Total missing values   : 29
Duplicate rows         : 6

The duplicate records:


     Student_ID            Name  Gender   Age  Study_Hours Attendance  Math  Science  English
77         1006    Vikram Menon  Female  19.0          2.3        79%  41.0     39.0     21.0
99         1006    Vikram Menon  Female  19.0          2.3        79%  41.0     39.0     21.0
66         1007     Kavya Kumar    Male  19.0          4.5        92%  55.0     57.0     64.0
14         1007     Kavya Kumar    Male  19.0          4.5        92%  55.0     57.0     64.0
89         1096       Pooja Rao    Male  19.0          4.4        71%  50.0     39.0     49.0
100        1096       Pooja Rao    Male  19.0          4.4        71%  50.0     39.0     49.0
25         1102  Divya Krishnan  Female  19.0          3.8        84%  59.0     55.0     36.0
20         1102  Divya Krishnan  Female  19.0          3.8        84%  59.0     55.0     36.0
32         1104  Karthik Sharma    Male  19.0          1.5       100%  39.0     35.0     36.0
82         1104  Karthik Sharma    Male  19.0          1.5  

Observations (before cleaning)
- 126 rows, 9 columns.
- Missing values in `Age`, `Study_Hours`, `Attendance`, `Math`, `Science`, `English`.
- 6 exact duplicate rows.
- `Attendance` is stored as text (e.g. `85%`) so calculations are not possible; `Age` is `float` because of NaN; `Gender` should be a category.

## B. Cleaning steps

In [5]:
# Step 1 - remove duplicate records
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (120, 9)


In [6]:
# Step 2 - correct data type: Attendance '85%' (text) -> 85.0 (number)
df["Attendance"] = pd.to_numeric(df["Attendance"].str.rstrip("%"))
print(df["Attendance"].dtype)

float64


In [7]:
# Step 3 - handle missing values
# Numeric columns -> fill with the column mean (rounded); Age -> fill with the median
for col in ["Study_Hours", "Attendance", "Math", "Science", "English"]:
    df[col] = df[col].fillna(round(df[col].mean(), 1))
df["Age"] = df["Age"].fillna(df["Age"].median())
print("Missing values left:", df.isnull().sum().sum())

Missing values left: 0


In [8]:
# Step 4 - correct remaining data types
df["Age"] = df["Age"].astype(int)          # float -> int
df["Gender"] = df["Gender"].astype("category")
df = df.reset_index(drop=True)
df.dtypes

Student_ID        int64
Name                str
Gender         category
Age               int64
Study_Hours     float64
Attendance      float64
Math            float64
Science         float64
English         float64
dtype: object

## C. After cleaning - dataset analysis

In [9]:
print("Shape:", df.shape)
df.info()

Shape: (120, 9)
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   Student_ID   120 non-null    int64   
 1   Name         120 non-null    str     
 2   Gender       120 non-null    category
 3   Age          120 non-null    int64   
 4   Study_Hours  120 non-null    float64 
 5   Attendance   120 non-null    float64 
 6   Math         120 non-null    float64 
 7   Science      120 non-null    float64 
 8   English      120 non-null    float64 
dtypes: category(1), float64(5), int64(2), str(1)
memory usage: 7.8 KB


In [10]:
print("Missing values:", df.isnull().sum().sum(), "| Duplicates:", df.duplicated().sum())
df.describe().round(2)

Missing values: 0 | Duplicates: 0


       Student_ID     Age  Study_Hours  Attendance    Math  Science  English
count      120.00  120.00       120.00       120.0  120.00   120.00   120.00
mean      1060.50   18.58         3.78        82.0   47.26    49.54    47.80
std         34.79    1.05         1.39         8.4   11.11    10.77    11.51
min       1001.00   17.00         0.80        55.0   22.00    17.00    21.00
25%       1030.75   18.00         2.80        77.0   39.00    43.00    40.00
50%       1060.50   19.00         3.80        82.0   47.30    49.50    47.80
75%       1090.25   19.00         4.72        87.0   54.00    57.00    55.25
max       1120.00   20.00         7.00       100.0   80.00    79.00    76.00

In [11]:
# Before vs After comparison
pd.DataFrame({"Before": [raw.shape[0], raw.isnull().sum().sum(), raw.duplicated().sum()],
              "After":  [df.shape[0], df.isnull().sum().sum(), df.duplicated().sum()]},
             index=["Rows", "Missing values", "Duplicate rows"])

                Before  After
Rows               126    120
Missing values      29      0
Duplicate rows       6      0

In [12]:
# Display the cleaned dataset and save it
df.to_csv("students_clean.csv", index=False)
print("Saved students_clean.csv")
df.head(10)

Saved students_clean.csv


   Student_ID              Name  Gender  Age  Study_Hours  Attendance  Math  Science  English
0        1052      Deepa Sharma  Female   19          2.9        66.0  36.0     50.0     54.0
1        1003        Swathi Rao  Female   18          3.1        89.0  43.0     37.0     43.0
2        1079    Prakash Sharma    Male   20          5.4        80.0  45.0     47.0     51.0
3        1010        Divya Nair    Male   19          1.4        76.0  22.0     29.0     50.0
4        1017    Keerthi Pillai    Male   17          6.2        74.0  60.0     62.0     47.0
5        1078     Ganesh Pillai    Male   20          5.2        68.0  60.0     43.0     50.0
6        1001  Prakash Krishnan  Female   20          4.3        69.0  36.0     48.0     47.8
7        1103       Nithya Nair  Female   18          2.5        87.0  39.0     58.0     39.0
8        1116        Swathi Rao  Female   20          7.0        84.0  47.3     67.0     75.0
9        1059     Swathi Pillai    Male   18          3.2   

## Explanation of the cleaning process
1. **Identified problems** using `info()`, `isnull().sum()` and `duplicated()`.
2. **Removed 6 duplicate rows** with `drop_duplicates()` because duplicates would count the same student twice.
3. **Converted `Attendance`** from text like `85%` to a number so it can be used in calculations.
4. **Filled missing values** instead of deleting rows, so that no student's data is lost: the column mean for marks, study hours and attendance, and the median for age (age is a whole number).
5. **Corrected data types** (`Age` to int, `Gender` to category). The final dataset has 120 rows and no missing or duplicate values.